# Residual-Stream Metric Tracking During Generation (PyTorch + Transformers)

This notebook demonstrates a **token-by-token generation pipeline** that:

1. Loads a decoder-only language model (default: `gpt2`).
2. Registers a `forward_hook` on the **final decoder block** to capture the residual stream hidden state $h_t$ right before unembedding.
3. Maintains a sliding context window buffer $\mathcal{W}$ with size $k=50$.
4. Computes four online geometry/dynamics metrics at each generation step:
   - **LID** (TwoNN)
   - **Jerk** (3rd finite difference norm)
   - **Curvature** (Menger curvature via Gram determinant)
   - **Energy** (Mahalanobis distance)
5. Logs per-token outputs and metrics to a CSV for post-hoc analysis.


In [ ]:
import csv
from collections import deque
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed


In [ ]:
# -----------------------------
# Configuration
# -----------------------------
MODEL_NAME = "gpt2"  # swap to a local larger model if available (e.g. Llama)
WINDOW_K = 50
MAX_NEW_TOKENS = 40
TEMPERATURE = 0.8
TOP_P = 0.95
DO_SAMPLE = True
SEED = 42
OUTPUT_CSV = Path("residual_metrics_log.csv")

PROMPTS = [
    {"type": "reasoning", "text": "If a train travels 120 km in 1.5 hours, what is its average speed in km/h? Explain briefly."},
    {"type": "reasoning", "text": "A rectangle has perimeter 30 and width 5. What is its area? Show your steps."},
    {"type": "creative", "text": "Write a short surreal scene where the moon negotiates with the ocean."},
    {"type": "creative", "text": "Compose a 4-line poem about forgotten keys and rainy sidewalks."},
]

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# -----------------------------
# Model + tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()


In [ ]:
# -----------------------------
# Metric utilities
# -----------------------------
def twonn_lid(states: torch.Tensor, eps: float = 1e-12) -> float:
    """Estimate Local Intrinsic Dimension using TwoNN over states in window."""
    n = states.shape[0]
    if n < 3:
        return float("nan")

    dists = torch.cdist(states, states, p=2)  # [n, n]
    dists.fill_diagonal_(float("inf"))

    vals, _ = torch.topk(dists, k=2, largest=False, dim=1)
    r1 = vals[:, 0].clamp_min(eps)
    r2 = vals[:, 1].clamp_min(eps)

    ratios = (r2 / r1).clamp_min(1.0 + eps)
    logs = torch.log(ratios)
    lid = 1.0 / logs.mean().item()
    return lid


def jerk_metric(trajectory: list[torch.Tensor]) -> float:
    """||h_t - 3h_{t-1} + 3h_{t-2} - h_{t-3}||_2"""
    if len(trajectory) < 4:
        return float("nan")
    h_t, h_t1, h_t2, h_t3 = trajectory[-1], trajectory[-2], trajectory[-3], trajectory[-4]
    jerk_vec = h_t - 3.0 * h_t1 + 3.0 * h_t2 - h_t3
    return torch.linalg.vector_norm(jerk_vec).item()


def menger_curvature_gram(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor, eps: float = 1e-12) -> float:
    """Menger curvature for triplet (a,b,c) using Gram determinant."""
    u = b - a
    v = c - a

    uu = torch.dot(u, u)
    uv = torch.dot(u, v)
    vv = torch.dot(v, v)

    det_g = (uu * vv - uv * uv).clamp_min(0.0)
    area = 0.5 * torch.sqrt(det_g)

    ab = torch.linalg.vector_norm(b - a)
    bc = torch.linalg.vector_norm(c - b)
    ac = torch.linalg.vector_norm(c - a)

    denom = (ab * bc * ac).clamp_min(eps)
    return (4.0 * area / denom).item()


def curvature_metric(trajectory: list[torch.Tensor]) -> float:
    if len(trajectory) < 3:
        return float("nan")
    return menger_curvature_gram(trajectory[-3], trajectory[-2], trajectory[-1])


def mahalanobis_energy(h_t: torch.Tensor, states: torch.Tensor, reg: float = 1e-4) -> float:
    """(h_t - mu)^T Sigma^{-1} (h_t - mu), where Sigma is from window states."""
    n, d = states.shape
    if n < 2:
        return float("nan")

    mu = states.mean(dim=0)
    centered = states - mu
    cov = (centered.T @ centered) / max(n - 1, 1)
    cov = cov + reg * torch.eye(d, device=states.device, dtype=states.dtype)
    cov_inv = torch.linalg.pinv(cov)

    diff = h_t - mu
    energy = diff @ cov_inv @ diff
    return energy.item()


In [ ]:
# -----------------------------
# Hook setup: final decoder block residual stream
# -----------------------------
residual_buffer = deque(maxlen=WINDOW_K)
latest_hidden = {"value": None}

# GPT-2 style decoder blocks live in model.transformer.h
# For Llama-style models you would typically use model.model.layers[-1]
final_block = model.transformer.h[-1]

def capture_final_residual(module, module_input, module_output):
    hidden_states = module_output[0] if isinstance(module_output, (tuple, list)) else module_output
    h_t = hidden_states[:, -1, :].detach().to(dtype=torch.float32).cpu().squeeze(0)
    latest_hidden["value"] = h_t

hook_handle = final_block.register_forward_hook(capture_final_residual)
print("Forward hook attached to final decoder block.")


In [ ]:
# -----------------------------
# Generation loop + online metric logging
# -----------------------------
fields = [
    "prompt_id",
    "prompt_type",
    "step",
    "token_id",
    "token_text",
    "lid_twonn",
    "jerk",
    "curvature",
    "energy",
]

all_rows = []

with torch.inference_mode():
    for prompt_id, item in enumerate(PROMPTS):
        prompt_type = item["type"]
        prompt_text = item["text"]

        encoded = tokenizer(prompt_text, return_tensors="pt").to(device)
        input_ids = encoded["input_ids"]
        attention_mask = encoded["attention_mask"]

        residual_buffer.clear()
        trajectory = []

        generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        print(f"\n--- Prompt {prompt_id} ({prompt_type}) ---")
        print(generated_text)

        for step in range(1, MAX_NEW_TOKENS + 1):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits[:, -1, :]

            if DO_SAMPLE:
                probs = torch.softmax(logits / TEMPERATURE, dim=-1)
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                cumsum = torch.cumsum(sorted_probs, dim=-1)
                nucleus = cumsum <= TOP_P
                nucleus[..., 0] = True

                filtered = torch.zeros_like(probs)
                filtered.scatter_(1, sorted_idx, sorted_probs * nucleus)
                filtered = filtered / filtered.sum(dim=-1, keepdim=True)
                next_token = torch.multinomial(filtered, num_samples=1)
            else:
                next_token = torch.argmax(logits, dim=-1, keepdim=True)

            h_t = latest_hidden["value"]
            if h_t is None:
                raise RuntimeError("Hook did not capture hidden state; check hook location.")

            residual_buffer.append(h_t)
            trajectory.append(h_t)

            window_states = torch.stack(list(residual_buffer), dim=0)

            lid_val = twonn_lid(window_states)
            jerk_val = jerk_metric(trajectory)
            curvature_val = curvature_metric(trajectory)
            energy_val = mahalanobis_energy(h_t, window_states)

            token_id = int(next_token.item())
            token_text = tokenizer.decode([token_id], skip_special_tokens=False)

            row = {
                "prompt_id": prompt_id,
                "prompt_type": prompt_type,
                "step": step,
                "token_id": token_id,
                "token_text": token_text,
                "lid_twonn": lid_val,
                "jerk": jerk_val,
                "curvature": curvature_val,
                "energy": energy_val,
            }
            all_rows.append(row)

            input_ids = torch.cat([input_ids, next_token], dim=1)
            next_attn = torch.ones((attention_mask.size(0), 1), device=device, dtype=attention_mask.dtype)
            attention_mask = torch.cat([attention_mask, next_attn], dim=1)

        decoded = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        print("\nGenerated completion:")
        print(decoded[len(generated_text):])

with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(all_rows)

print(f"\nSaved {len(all_rows)} rows to: {OUTPUT_CSV.resolve()}")


In [ ]:
# Always remove hooks when done
hook_handle.remove()
print("Hook removed.")


## Notes

- The code uses `torch.inference_mode()` and explicit `.detach()` inside the hook to avoid gradient storage.
- For very large models (e.g., Llama-3-8B), ensure adequate VRAM or run with quantization.
- If you switch architecture families, update the final-layer module path used by `register_forward_hook`.
